
# Tool Integration Completeness Evaluation

This notebook evaluates the orchestrator's capability to integrate with multiple distinct tools, validate against schemas, handle execution failures, and gracefully resolve ambiguous/flawed inputs before triggering raw API integrations.


In [1]:

import pandas as pd
import json
import asyncio
import time
import random
from pprint import pprint
import matplotlib.pyplot as plt
import sys
import os

# Add backend directory to path
sys.path.insert(0, os.path.abspath('.'))

# Import actual orchestrator pipeline
# Note: we wrap run_orchestrator in our simulator to grade parameter constraints strictly
from app.graph.workflow import run_orchestrator

def print_break():
    print("=" * 80)
    
print("✅ Libraries and Orchestrator imported successfully.")


✅ Libraries and Orchestrator imported successfully.



---
## 1. Simulated Tool Schemas & Executions
Defining the constraints for Flashcards, Concept Explainer, Note Maker, and Quiz Me tools. 
We'll map valid execution protocols, inject latency, and define error thresholds.


In [2]:

# Strict schemas to evaluate the LLM against
TOOL_REGISTRY = {
    "flashcards": {
        "required": ["topic"],
        "optional": {"count": 5, "subject": "general"},
        "enums": {"difficulty": ["easy", "medium", "hard"]}
    },
    "concept_explainer": {
        "required": ["concept_to_explain"],
        "optional": {"current_topic": "general"},
        "enums": {"desired_depth": ["basic", "intermediate", "advanced", "comprehensive"]}
    },
    "note_maker": {
        "required": ["topic"],
        "optional": {"subject": "general"},
        "enums": {"note_taking_style": ["outline", "bullet_points", "narrative", "structured"]}
    },
    "quiz_me": {
        "required": ["topic"],
        "optional": {"count": 5},
        "enums": {"difficulty": ["easy", "medium", "hard"]}
    }
}
print("Registry Simulated Context Loaded.")


Registry Simulated Context Loaded.



---
## 2. Evaluation Dataset
Includes varied tests: Correct triggers, multi-tool ambiguity, missing parameters, and explicit invalid parameters to check error traps.


In [3]:

EVAL_DATASET = [
    {
        "id": "success_01",
        "input": "Make me 10 hard flashcards about anatomy for biology.",
        "expected_tool": "flashcards",
        "expected_behavior": "execute_success",
        "simulate_api_fail": False
    },
    {
        "id": "success_02",
        "input": "Explain the concept of inflation deeply.",
        "expected_tool": "concept_explainer",
        "expected_behavior": "execute_success",
        "simulate_api_fail": False
    },
    {
        "id": "error_schema_01",
        "input": "Create some notes on geography.",
        "expected_tool": "note_maker",
        "expected_behavior": "validation_failure", # Implicitly lacks the style or required user profiles
        "simulate_api_fail": False
    },
    {
        "id": "error_api_fail_01",
        "input": "Quiz me on quantum physics, extremely hard mode.",
        "expected_tool": "quiz_me",
        "expected_behavior": "graceful_fail", # We inject a 500 API failure and ensure orchestrator catches it
        "simulate_api_fail": True
    },
]
print(f"Loaded {len(EVAL_DATASET)} integration evaluation cases.")


Loaded 4 integration evaluation cases.



---
## 3. Schema Validator & Tool Executor Simulator
Validates parameters exactly prior to simulated execution to check if the orchestrator bypassed rules.


In [4]:

def validate_params(params: dict, schema: dict) -> tuple[bool, list]:
    errors = []
    # Check Required
    for req in schema.get("required", []):
        if req not in params:
            errors.append(f"Missing required parameter: {req}")
            
    # Check Enums
    for enum_key, allowed_vals in schema.get("enums", {}).items():
        if enum_key in params and params[enum_key] not in allowed_vals:
            # We forgive string capitalization
            if isinstance(params[enum_key], str) and params[enum_key].lower() in [v.lower() for v in allowed_vals]:
                continue
            errors.append(f"Enum violation on {enum_key}: '{params[enum_key]}' not in {allowed_vals}")
            
    return len(errors) == 0, errors

def execute_tool_simulated(tool_name: str, params: dict, force_fail: bool = False):
    start_time = time.time()
    latency = random.uniform(0.5, 1.5)
    time.sleep(latency) # Simulate execution latency
    
    if force_fail:
        return {"success": False, "error": "500 Internal Server Error: Tool Endpoint Down", "latency": latency}
        
    return {"success": True, "response_payload": {"message": f"Successfully executed {tool_name}"}, "latency": latency}



---
## 4. Integration Test Runner
For each case, run the orchestrator, scrape the output, run our strict parameter validator, and evaluate error survivability.


In [5]:

async def run_integration_tests():
    results = []
    for test in EVAL_DATASET:
        print(f"Testing: {test['id']} | Action: {test['expected_behavior']}")
        
        # 1. ORCHESTRATE
        t0 = time.time()
        try:
            state = await run_orchestrator(
                message=test["input"], 
                session_id=f"integration_eval_{test['id']}", 
                student_id="student_eval"
            )
            predicted_tool = state.get("selected_tool", "")
            params = state.get("validated_params", {})
            orchestrator_error = state.get("error_message")
            exec_success = state.get("execution_success", False)
        except Exception as e:
            results.append({
                "test_id": test["id"], "input": test["input"], "expected_tool": test["expected_tool"],
                "predicted_tool": "CRASH", "validation_status": "FAIL", "execution_success": False,
                "error_handling_score": 0.0, "latency": time.time() - t0, "errors": str(e)
            })
            continue

        # 2. VALIDATE THE ORCHESTRATOR'S EXTRACTED PARAMS 
        if predicted_tool in TOOL_REGISTRY:
            is_valid, val_errors = validate_params(params, TOOL_REGISTRY[predicted_tool])
        else:
            is_valid, val_errors = False, ["Predicted tool not in Registry"]

        # 3. ASSESS ERROR HANDLING
        error_handling_score = 0.0
        if test["simulate_api_fail"]:
            # Emulate the fail behavior
            sim_result = execute_tool_simulated(predicted_tool, params, force_fail=True)
            # if orchestrator handled it natively without crashing its loop:
            if not getattr(state, "Exception_Occured", False):
                 error_handling_score = 1.0 
        elif not is_valid:
            if orchestrator_error or not exec_success:
                 error_handling_score = 1.0 # Successfully caught schema violation before execution
        else:
            if is_valid and predicted_tool == test["expected_tool"]:
                error_handling_score = 1.0 # Normal valid path
                
        # Calculate response structure correctness
        response_quality = sum([
            1 if predicted_tool == test["expected_tool"] else 0,
            1 if is_valid else 0,
            1 if error_handling_score > 0.5 else 0
        ]) / 3
            
        results.append({
            "test_id": test["id"],
            "input": test["input"],
            "expected_tool": test["expected_tool"],
            "predicted_tool": predicted_tool,
            "validation_status": "PASS" if is_valid else "FAIL",
            "execution_success": exec_success,
            "response_quality": round(response_quality, 2),
            "error_handling_score": error_handling_score,
            "val_errors": val_errors
        })
        
    return pd.DataFrame(results)

df_results = await run_integration_tests()
display(df_results)


Testing: success_01 | Action: execute_success
2026-04-18 03:29:50,366 INFO {"stage": "Reasoner", "level": "INFO", "message": "short-circuit: intent=flashcards, conf=0.9", "data": {"source": "rule"}}
2026-04-18 03:29:50,369 INFO {"stage": "Tool Selector", "level": "INFO", "message": "selected=flashcards conf=0.63 (ranker=0.44, reasoning=0.90) via ranker; top3=['flashcards', 'mnemonic_generator', 'anchor_chart_maker']; dropped=0 schema-incompatible", "data": {"reason": "keywords ['flashcards', 'for']; intent match 'memorize'; +intent-match boost (+0.30)"}}
2026-04-18 03:29:51,441 INFO {"stage": "Extractor", "level": "INFO", "message": "extracted 4 explicit params for flashcards", "data": {"params": ["topic", "count", "difficulty", "subject"]}}
2026-04-18 03:29:51,444 INFO {"stage": "Personalization", "level": "INFO", "message": "plan: mastery L5, emotion=confused, style=direct → difficulty=easy, depth=basic, count=3", "data": {"reasons": ["mastery L5 → difficulty=medium, depth=intermedia

,test_id,input,expected_tool,predicted_tool,validation_status,execution_success,response_quality,error_handling_score,val_errors
0,success_01,Make me 10 hard flashcards about anatomy for b...,flashcards,flashcards,PASS,True,1.00,1.0,[]
1,success_02,Explain the concept of inflation deeply.,concept_explainer,concept_explainer,PASS,True,1.00,1.0,[]
2,error_schema_01,Create some notes on geography.,note_maker,anchor_chart_maker,FAIL,True,0.00,0.0,[Predicted tool not in Registry]
3,error_api_fail_01,"Quiz me on quantum physics, extremely hard mode.",quiz_me,quiz_me,FAIL,True,0.67,1.0,[Enum violation on difficulty: 'extremely hard...



---
## 5. Metrics & Failure Analysis
Output the tool integration success rate, error handling success limits, and failure traces.


In [6]:

# Metrics computation
print_break()
print("🏆 TOOL INTEGRATION SCORING METRICS")
success_rate = len(df_results[df_results['predicted_tool'] == df_results['expected_tool']]) / len(df_results)
print(f"Tool Selection Accuracy:     {success_rate * 100:.1f}%")

schema_compliance = len(df_results[df_results['validation_status'] == 'PASS']) / len(df_results)
print(f"Schema Compliance Rate:      {schema_compliance * 100:.1f}%")

error_rate = df_results['error_handling_score'].mean()
print(f"Error Handling Survivability:{error_rate * 100:.1f}%")

overall_score = df_results['response_quality'].mean()
print(f"Overall Integration Score:   {overall_score * 100:.1f}%")


print_break()
print("🚨 FAILURE ANALYSIS")

failures = df_results[df_results['response_quality'] < 1.0]
if failures.empty:
    print("Perfect Tool Integration! No execution leaks or schema breaks detected.")
else:
    for _, row in failures.iterrows():
        print(f"🔹 ID: {row['test_id']} | Input: '{row['input']}'")
        if row['predicted_tool'] != row['expected_tool']:
            print(f"   [!] WRONG TOOL: Expected '{row['expected_tool']}', Got '{row['predicted_tool']}'")
        if row['validation_status'] == "FAIL":
            print(f"   [!] SCHEMA BREAK: {row['val_errors']}")
        if row['error_handling_score'] == 0.0:
            print(f"   [!] UNGRACEFUL FAIL: Orchestrator failed to trap or correct the API/Schema failure state.")
        print("")


🏆 TOOL INTEGRATION SCORING METRICS
Tool Selection Accuracy:     75.0%
Schema Compliance Rate:      50.0%
Error Handling Survivability:75.0%
Overall Integration Score:   66.8%
🚨 FAILURE ANALYSIS
🔹 ID: error_schema_01 | Input: 'Create some notes on geography.'
   [!] WRONG TOOL: Expected 'note_maker', Got 'anchor_chart_maker'
   [!] SCHEMA BREAK: ['Predicted tool not in Registry']
   [!] UNGRACEFUL FAIL: Orchestrator failed to trap or correct the API/Schema failure state.

🔹 ID: error_api_fail_01 | Input: 'Quiz me on quantum physics, extremely hard mode.'
   [!] SCHEMA BREAK: ["Enum violation on difficulty: 'extremely hard' not in ['easy', 'medium', 'hard']"]




---
## 6. Pipeline Trace View
Deep log of the first execution block.


In [7]:

first = df_results.iloc[0]
print("🔍 INTEGRATION PIPELINE TRACE")
print(f"1. INPUT          → {first['input']}")
print(f"2. ROUTER         → Expected: {first['expected_tool']} | Selected: {first['predicted_tool']}")
print(f"3. VALIDATION     → Passed local strict checker: {first['validation_status']}")
if first['val_errors']:
    print(f"                    (Errors: {first['val_errors']})")
print(f"4. EXECUTION      → Successful API trigger: {first['execution_success']}")
print(f"5. ERROR HANDLER  → Resilience Score: {first['error_handling_score']}")


🔍 INTEGRATION PIPELINE TRACE
1. INPUT          → Make me 10 hard flashcards about anatomy for biology.
2. ROUTER         → Expected: flashcards | Selected: flashcards
3. VALIDATION     → Passed local strict checker: PASS
4. EXECUTION      → Successful API trigger: True
5. ERROR HANDLER  → Resilience Score: 1.0
